# Sugeno Integral

## Learning objectives

By the end of this notebook, you should be able to:

- Explain why the **Sugeno integral** is a nonlinear aggregation operator.
- Describe the roles of the input values and the **fuzzy measure** in the aggregation.
- Follow the sorting/subset construction used when computing a discrete Sugeno integral.
- Interpret how changing either the inputs or the fuzzy measure can change the aggregate result.
- Connect the Python implementation to the mathematical steps of the Sugeno integral.

In [ ]:
# Step 1: follow the variables below and connect each computation to the Sugeno-integral procedure.
%matplotlib inline

# Sugeno integral

Lets establish some notation for below

 * $N$ sources with real-valued inputs
   * sources
     * $X = \{ x_1, x_2, ..., x_N \} $
   * inputs 
     * $h(\{x_i\}) \in \Re$ (or $h_i$ short hand notation)
   * input sort ($\pi$ function)
     * $h_{\pi{(1)}} \geq h_{\pi{(2)}} \geq ... \geq h_{\pi{(N)}}$
 * Fuzzy measure (for finite/discrete $X$)
   * Boundary conditions
     * $g(A) \geq 0, \forall A \in 2^X$ (and often $g(X)=1$)
   * Monotonicity conditions
     * for $A, B \in 2^{X}$, $g(A \cup B) \geq g(A)$ and $g(A \cup B) \geq g(B)$
 * Sugeno integral
   * $ S_g(h) = \int{ h \circ g } = \vee_{i=1}^{N}{ \left( h_{\pi{(i)}} \wedge g(\{x_{\pi(1)},...,x_{\pi(i)}\}) \right) }$

In [ ]:
# Step 2: follow the variables below and connect each computation to the Sugeno-integral procedure.
# lets import some libs
import numpy as np
import itertools
import matplotlib
import matplotlib.pyplot as plt

# lets make our sugeno integral class
class SugenoIntegral:

    def __init__(self):
        """
            init function
        """
        
        self.N = 0    
        self.fm = []  
        self.g = []   
        
    def evaluate(self, x):
        """
            evaluate the Sugeno integral
        """

        hs = np.zeros(x.size)
        gs = np.zeros(x.size)
        
        print("Input:",x)
                
        # do our sort
        pi_i = np.argsort(x)[::-1] + 1
        
        print("Sort:",pi_i)
        
        # do the first calculation
        h = x[pi_i[0] - 1]
        g = self.fm[ str(pi_i[:1]) ]
        o = min( h , g )
        print("[h,g] Step 1 :",h,g)
        print(str(pi_i[:1]))
        hs[0] = h
        gs[0] = g
        
        # do the other N-1 terms
        for i in range(1, self.N):
            h = x[pi_i[i] - 1]            
            g = self.fm[str(np.sort(pi_i[:i + 1]))]
            hs[i] = h
            gs[i] = g
            print("[h,g] Step",i+1,":",h,g)
            print(str(np.sort(pi_i[:i + 1])))
            # our calculation, namely, max of the mins
            o = max( o, min( h, g ) )
            
        return o, hs, gs

    def get_keys_index(self):
        """
            sets up a dictionary for referencing the FM
            :return: keys to the dictionary
        """

        vls = np.arange(1, self.N + 1)
        count = 0
        Lattice = {}
        for i in range(0, self.N):
            Lattice[str(np.array([vls[i]]))] = count
            count = count + 1
        for i in range(2, self.N + 1):
            A = np.array(list(itertools.combinations(vls, i)))
            for latt_pt in A:
                Lattice[str(latt_pt)] = count
                count = count + 1
        return Lattice    
    
    def produce_lattice(self):
        """
            makes a nice little ole data structure for us to index our fuzzy measure 
        """

        index_keys = self.get_keys_index()
        Lattice = {}
        for key in index_keys.keys():
            Lattice[key] = self.g[index_keys[key]]
        return Lattice

Now, lets use it

In [ ]:
# Step 3: follow the variables below and connect each computation to the Sugeno-integral procedure.
# create our integral
sugeno = SugenoIntegral()

# lets go with 4 inputs in our example
sugeno.N = 4

# fuzzy measure
#fm = [ x1 x2 x3 x4 x12 x13 x14 x23 x24 x34 x123 x124 x134 x234 x1234 ]
#fm = np.asarray( [ 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1 ] ) # min
#fm = np.asarray( [ 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1 ] ) # max
fm = np.asarray( [ 1/4, 1/4, 1/4, 1/4, 1/2, 1/2, 1/2, 1/2, 1/2, 1/2, 3/4, 3/4, 3/4, 3/4, 1 ] ) 
sugeno.g = fm

# convert it into a data structure that is a little nicer to work with
sugeno.fm = sugeno.produce_lattice()
print("############################################")
print("Here is the fuzzy measure")
print("############################################")
print(sugeno.fm)

# do for one sample
print("############################################")
print("Here is evaluation of the fuzzy integral")
print("############################################")
o, hs, gs = sugeno.evaluate(np.asarray([0.8,0.6,0.3,0.4]))
print( o )

Plot that best pessimistic agreement 

In [ ]:
# Step 4: follow the variables below and connect each computation to the Sugeno-integral procedure.
plt.stem(hs, markerfmt='ko')
plt.plot(hs,'k')
plt.plot(gs,'r')
plt.title('h and g')
plt.grid()
plt.show()

### Before moving on

Make sure you can answer these questions without looking at the code:

- Why does the Sugeno integral need a fuzzy measure rather than just one weight per input?
- Where do the minimum and maximum operations enter the calculation?
- Why does the ordering of the inputs matter?
- If one fuzzy-measure value changes, which parts of the calculation could be affected?

A useful test of understanding is to change one input value, predict how the ordering and candidate values will change, and then rerun the notebook to check your prediction.